# B2S 07 - AndinaLog Flota Eventos

Conversión Bronze JSON a Silver de eventos de flota. Se documenta el aplanamiento de objetos padre a eventos hijos y se conserva la trazabilidad de cada posición del arreglo.

In [1]:
import json
import os
import platform
from datetime import datetime, timezone
from pathlib import Path
from zoneinfo import ZoneInfo
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 100)

def find_root():
    candidates = []
    if os.getenv('ANDINALOG_ROOT'):
        candidates.append(Path(os.environ['ANDINALOG_ROOT']))
    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])
    for candidate in candidates:
        if (candidate / 'datos' / 'bronze').is_dir():
            return candidate
    raise FileNotFoundError('No se encontró datos/bronze')

ROOT = find_root()
EXECUTED_AT_UTC = datetime.now(timezone.utc).isoformat()
CONFIG = {
    'entidad': 'evento de telemetría de camión',
    'granularidad_padre': 'un objeto por camion_id con lista eventos',
    'granularidad_hijo': 'una fila aplanada por evento_id y posicion del arreglo',
    'rutas': {
        'bronze': 'datos/bronze/andinalog_flota_eventos.json',
        'silver': 'datos/silver/andinalog_flota_eventos_silver.csv',
        'quarantine': 'datos/quarantine/andinalog_flota_eventos_quarantine.csv',
        'informe': 'informes/bronze_silver/Informe_B2S_07_AndinaLog_Flota_Eventos.md',
        'notebook': 'notebooks/bronze_silver/07_flota_eventos/B2S_07_Flota_Eventos.ipynb',
        'flota_silver': 'datos/silver/andinalog_flota_silver.csv',
        'wms_silver': 'datos/silver/andinalog_wms_orders_silver.csv',
        'warehouse_informe': 'informes/bronze_silver/Informe_B2S_06_Warehouse_Costs.md',
        'warehouse_notebook': 'notebooks/bronze_silver/06_warehouse_costs/B2S_06_Andinalog_Warehouse_Costs.ipynb',
        'warehouse_quarantine': 'datos/quarantine/andinalog_warehouse_costs_quarantine.csv',
        'warehouse_silver': 'datos/silver/andinalog_warehouse_costs_silver.csv'
    },
    'lectura': {'encoding': 'utf-8'},
    'tipos_validos': ['ALERTA_TEMP_CADENA_FRIO', 'MANTENIMIENTO_PROGRAMADO', 'EXCESO_VELOCIDAD', 'GEOCERCA_SALIDA', 'FALLA_MOTOR'],
    'severidades_validas': ['Baja', 'Media', 'Alta'],
    'zona_sin_zona': 'America/La_Paz',
    'destino_timestamp': 'UTC',
    'imputaciones': {},
    'regla_duplicados': 'Duplicado exacto: una fila Silver y copias con trazabilidad en cuarentena; conflicto: todas las filas en cuarentena'
}
PATHS = {key: ROOT / value for key, value in CONFIG['rutas'].items()}
for key in ['silver', 'quarantine', 'informe']:
    PATHS[key].parent.mkdir(parents=True, exist_ok=True)
print('Raíz detectada:', ROOT)
print('Configuración centralizada para: eventos de flota, B2S 07')
print('Fecha de ejecución UTC:', EXECUTED_AT_UTC)

Raíz detectada: C:\Users\remrodri\Github\practicasNotebookColab\proyecto-integradorV2
Configuración centralizada para: eventos de flota, B2S 07
Fecha de ejecución UTC: 2026-09-24T22:52:11.099270+00:00


In [2]:
with PATHS['bronze'].open(encoding=CONFIG['lectura']['encoding']) as f:
    raw = json.load(f)
padres = raw.get('camiones', [])
filas = []
for padre in padres:
    for posicion, evento in enumerate(padre.get('eventos', []), start=1):
        fila = {'camion_id': padre.get('camion_id'), 'posicion_evento': posicion, 'sistema': raw.get('sistema'), 'fecha_exportacion': raw.get('fecha_exportacion'), 'evento_id': evento.get('evento_id'), 'timestamp': evento.get('timestamp'), 'tipo': evento.get('tipo'), 'severidad': evento.get('severidad'), 'valor_lectura': evento.get('valor_lectura'), 'reconocido': evento.get('reconocido'), 'umbral_temp_cabina_c': padre.get('config', {}).get('umbral_temp_cabina_c'), 'geocerca_radio_km': padre.get('config', {}).get('geocerca_radio_km'), 'ultimo_mantenimiento': padre.get('config', {}).get('ultimo_mantenimiento')}
        filas.append(fila)
bronze = pd.DataFrame(filas)
bronze.insert(0, 'evento_tec_id', range(1, len(bronze) + 1))
flota_aux = pd.read_csv(PATHS['flota_silver'], dtype=str, keep_default_na=False)
perfil = {'objetos_padre': len(padres), 'eventos_hijos': len(bronze), 'padres_sin_eventos': sum(not p.get('eventos') for p in padres), 'sistema': raw.get('sistema'), 'fecha_exportacion': raw.get('fecha_exportacion'), 'camiones_observados': sorted(bronze['camion_id'].dropna().unique().tolist()), 'tipos_observados': sorted(bronze['tipo'].dropna().unique().tolist()), 'severidades_observadas': sorted(bronze['severidad'].dropna().unique().tolist()), 'duplicados_evento_id': int(bronze['evento_id'].duplicated(keep=False).sum()), 'valores_reconocido_ausentes': int(bronze['reconocido'].isna().sum()), 'lecturas_textuales': int(bronze['valor_lectura'].map(lambda v: isinstance(v, str)).sum()), 'flota_silver_camiones': sorted(flota_aux['camion_id'].dropna().unique().tolist())}
print('Perfil JSON y auxiliar:')
print(perfil)
print(bronze.head(5).to_string(index=False))

Perfil JSON y auxiliar:
{'objetos_padre': 30, 'eventos_hijos': 192, 'padres_sin_eventos': 0, 'sistema': 'TELEMATICA-AndinaLog', 'fecha_exportacion': '2026-08-30T00:00:00', 'camiones_observados': ['CAM-01', 'CAM-02', 'CAM-03', 'CAM-04', 'CAM-05', 'CAM-06', 'CAM-07', 'CAM-08', 'CAM-09', 'CAM-10', 'CAM-11', 'CAM-12', 'CAM-13', 'CAM-14', 'CAM-15', 'CAM-16', 'CAM-17', 'CAM-18', 'CAM-19', 'CAM-20', 'CAM-21', 'CAM-22', 'CAM-23', 'CAM-24', 'CAM-25', 'CAM-26', 'CAM-27', 'CAM-28', 'CAM-29', 'CAM-30'], 'tipos_observados': ['ALERTA_TEMP_CADENA_FRIO', 'EXCESO_VELOCIDAD', 'FALLA_MOTOR', 'GEOCERCA_SALIDA', 'MANTENIMIENTO_PROGRAMADO'], 'severidades_observadas': ['Alta', 'Baja', 'Media'], 'duplicados_evento_id': 16, 'valores_reconocido_ausentes': 22, 'lecturas_textuales': 10, 'flota_silver_camiones': ['CAM-02', 'CAM-03', 'CAM-04', 'CAM-05', 'CAM-06', 'CAM-07', 'CAM-08', 'CAM-09', 'CAM-10', 'CAM-11', 'CAM-12', 'CAM-13', 'CAM-14', 'CAM-15', 'CAM-16', 'CAM-17', 'CAM-18', 'CAM-19', 'CAM-20', 'CAM-21', 'CAM

In [3]:
def preparar(df):
    out = df.copy()
    for col in ['camion_id', 'evento_id', 'timestamp', 'tipo', 'severidad', 'valor_lectura', 'reconocido']:
        out[col + '_original'] = out[col]
    out['errores_bloqueantes'] = ''
    out['motivos_transformacion'] = ''
    out['motivos_imputacion'] = ''
    out['fue_transformada'] = False
    out['fue_imputada'] = False
    return out

def add_error(df, mask, motivo):
    out = df.copy()
    mask = mask.fillna(False)
    current = out.loc[mask, 'errores_bloqueantes']
    out.loc[mask, 'errores_bloqueantes'] = np.where(current.eq(''), motivo, current + ';' + motivo)
    return out

def normalizar_campos(df):
    out = df.copy()
    out['camion_id'] = out['camion_id'].astype('string').str.strip()
    out['evento_id'] = out['evento_id'].astype('string').str.strip()
    out['tipo'] = out['tipo'].astype('string').str.strip()
    out['severidad'] = out['severidad'].astype('string').str.strip()
    out['fue_transformada'] |= out['camion_id'].ne(out['camion_id_original']) | out['evento_id'].ne(out['evento_id_original'])
    return out

def convertir_valores(df):
    out = df.copy()
    out['valor_lectura'] = pd.to_numeric(out['valor_lectura'].replace('', pd.NA), errors='coerce')
    no_aplica = out['valor_lectura_original'].astype('string').str.strip().str.upper().eq('N/D')
    conversion_invalida = out['valor_lectura'].isna() & out['valor_lectura_original'].ne('') & ~no_aplica
    out['valor_lectura_no_aplica'] = no_aplica
    centinela = out['valor_lectura'].isin([-999])
    out['valor_lectura_conversion_invalida'] = conversion_invalida
    out['valor_lectura_centinela_detectado'] = centinela
    out = add_error(out, conversion_invalida | centinela, 'valor_lectura:conversion_invalida_o_centinela')
    out['reconocido_normalizado'] = out['reconocido'].map({True: True, False: False, 'true': True, 'false': False, 'True': True, 'False': False})
    out['reconocido_ausente'] = out['reconocido'].isna() | out['reconocido'].eq('')
    out['fue_transformada'] |= out['valor_lectura'].astype('string').ne(out['valor_lectura_original'].astype('string'))
    return out

def convertir_timestamps(df):
    out = df.copy()
    original = out['timestamp'].astype('string').str.strip()
    iso = pd.to_datetime(original, format='%Y-%m-%dT%H:%M:%S', errors='coerce')
    slash = pd.to_datetime(original, format='%d/%m/%Y %H:%M', errors='coerce')
    local = iso.fillna(slash)
    out['timestamp_utc'] = local.dt.tz_localize(ZoneInfo(CONFIG['zona_sin_zona'])).dt.tz_convert('UTC')
    out['timestamp_tratado'] = out['timestamp_utc'].dt.strftime('%Y-%m-%dT%H:%M:%S%z')
    out['timestamp_formato_detectado'] = np.where(original.eq(''), 'ausente', np.where(iso.notna(), 'ISO', np.where(slash.notna(), 'DD/MM/YYYY HH:MM', 'invalido')))
    out['timestamp_transformado'] = out['timestamp_formato_detectado'].ne('ISO') & out['timestamp_tratado'].notna()
    invalid = original.ne('') & local.isna()
    out['timestamp_conversion_invalida'] = invalid
    out = add_error(out, invalid, 'timestamp:fecha_hora_invalida')
    out['fue_transformada'] |= out['timestamp_transformado']
    return out

def validar_dominio(df):
    out = df.copy()
    tipo_invalido = ~out['tipo'].isin(CONFIG['tipos_validos'])
    severidad_invalida = ~out['severidad'].isin(CONFIG['severidades_validas'])
    camion_informativo = ~out['camion_id'].isin(set(flota_aux['camion_id'].str.strip()))
    out['tipo_valido'] = ~tipo_invalido
    out['severidad_valida'] = ~severidad_invalida
    out['camion_corresponde_flota_silver'] = ~camion_informativo
    out = add_error(out, tipo_invalido, 'tipo:categoria_no_reconocida')
    out = add_error(out, severidad_invalida, 'severidad:categoria_no_reconocida')
    return out

def controlar_duplicados(df):
    out = df.copy()
    event_cols = ['camion_id', 'evento_id', 'timestamp_tratado', 'tipo', 'severidad', 'valor_lectura', 'reconocido_normalizado']
    comparable = out[event_cols].copy()
    comparable['reconocido_normalizado'] = comparable['reconocido_normalizado'].fillna('AUSENTE')
    out['duplicado_evento_id'] = out['evento_id'].duplicated(keep=False)
    exact = pd.concat([out[['camion_id', 'evento_id']], comparable.drop(columns=['camion_id', 'evento_id'])], axis=1).duplicated(keep=False)
    out['duplicado_exacto'] = exact
    out['conflicto_evento'] = out['duplicado_evento_id'] & ~exact
    out['copia_duplicada'] = exact & ~pd.concat([out[['camion_id', 'evento_id']], comparable.drop(columns=['camion_id', 'evento_id'])], axis=1).duplicated(keep='first')
    out = add_error(out, out['conflicto_evento'], 'evento_id:duplicado_conflictivo')
    out = add_error(out, out['copia_duplicada'], 'evento_id:duplicado_exacto_copia_resuelta')
    out['motivos_transformacion'] = np.where(out['copia_duplicada'], 'duplicado_exacto:copia_conservada_con_trazabilidad', out['motivos_transformacion'])
    return out

def calcular_calidad(df):
    out = df.copy()
    out['calidad_motivo'] = out['errores_bloqueantes'].replace('', 'sin_incidencias')
    out['calidad_estado'] = np.where(out['errores_bloqueantes'].eq(''), 'valida', 'cuarentena')
    out['conteo_transformaciones'] = out['fue_transformada'].fillna(False).astype(int)
    out['conteo_imputaciones'] = out['fue_imputada'].fillna(False).astype(int)
    out['imputacion_metodo'] = ''
    out['imputacion_motivo'] = ''
    return out

work = bronze.pipe(preparar).pipe(normalizar_campos).pipe(convertir_valores).pipe(convertir_timestamps).pipe(validar_dominio).pipe(controlar_duplicados).pipe(calcular_calidad)
silver = work.loc[work['errores_bloqueantes'].eq('') & ~work['copia_duplicada']].copy()
quarantine = work.loc[work['errores_bloqueantes'].ne('') | work['copia_duplicada']].copy()
print('Silver', len(silver), 'Cuarentena', len(quarantine))
print('Motivos cuarentena:', quarantine['errores_bloqueantes'].value_counts().to_dict())
print('Silver estados:', silver['calidad_estado'].value_counts().to_dict())
print('Cuarentena estados:', quarantine['calidad_estado'].value_counts().to_dict())

Silver 184 Cuarentena 8
Motivos cuarentena: {'evento_id:duplicado_exacto_copia_resuelta': 8}
Silver estados: {'valida': 184}
Cuarentena estados: {'cuarentena': 8}


In [4]:
silver.to_csv(PATHS['silver'], index=False, encoding='utf-8')
quarantine.to_csv(PATHS['quarantine'], index=False, encoding='utf-8')
assert len(bronze) == len(silver) + len(quarantine)
assert set(silver['evento_tec_id']).isdisjoint(set(quarantine['evento_tec_id']))
assert len(padres) == bronze['camion_id'].nunique()
assert len(bronze) == perfil['eventos_hijos']
assert silver['errores_bloqueantes'].eq('').all()
assert silver['evento_id'].is_unique
assert silver['timestamp_utc'].notna().all()
assert silver['timestamp_utc'].dt.tz_convert('UTC').notna().all()
print('Controles de父子, trazabilidad, duplicados y timestamps: OK')
print(pd.DataFrame({'objetos_padre': [len(padres)], 'eventos_hijos': [len(bronze)], 'silver': [len(silver)], 'quarantine': [len(quarantine)]}))

Controles de父子, trazabilidad, duplicados y timestamps: OK
   objetos_padre  eventos_hijos  silver  quarantine
0             30            192     184           8


In [5]:
report = [
    '# Informe B2S 07 - AndinaLog Flota Eventos', '',
    '## Objetivo, entidad y granularidad',
    'Conversión auditada de eventos de telemetría desde un JSON padre-hijo hacia Silver.',
    f"- Entidad: {CONFIG['entidad']}.",
    f"- Granularidad padre: {CONFIG['granularidad_padre']}.",
    f"- Granularidad hija: {CONFIG['granularidad_hijo']}.", '',
    '## Perfil Bronze de esta ejecución',
    f"- Sistema: {perfil['sistema']}.",
    f"- Fecha de exportación: {perfil['fecha_exportacion']}.",
    f"- Objetos padre: {perfil['objetos_padre']}.",
    f"- Eventos hijos: {perfil['eventos_hijos']}.",
    f"- Padres sin eventos: {perfil['padres_sin_eventos']}.",
    f"- Filas con evento_id duplicado: {perfil['duplicados_evento_id']}.",
    f"- Valores reconocido ausentes: {perfil['valores_reconocido_ausentes']}.",
    f"- Lecturas inicialmente textuales: {perfil['lecturas_textuales']}.", '',
    '## Reglas y transformaciones',
    '- Se aplana cada evento hijo y se conserva camion_id, posicion_evento y configuración padre.',
    '- Los timestamps sin zona se interpretan en America/La_Paz y se convierten a UTC.',
    '- Se aceptan ISO y DD/MM/YYYY HH:MM; las conversiones quedan auditadas.',
    '- Se normalizan valor_lectura y booleanos solo desde representaciones observadas.',
    '- Duplicados exactos se resuelven con una fila Silver y copias trazables en cuarentena.',
    '- La ausencia de reconocido se conserva como AUSENTE y no se convierte en False.',
    '- N/D en valor_lectura se conserva como no_aplica y no se convierte en cero ni en error.', '',
    '## Integridad referencial',
    '- Los camiones se comparan con flota Silver como referencia informativa; no se bloquea por falta de correspondencia.',
    '- Los tipos y severidades se validan contra catálogos declarados.',
    '- No se usan datos personales.', '',
    '## Resultado y conciliación',
    f"- Silver: {len(silver)} filas.",
    f"- Cuarentena: {len(quarantine)} filas.",
    f"- Conciliación de granularidad: {perfil['objetos_padre']} padres y {perfil['eventos_hijos']} eventos = {len(silver)} Silver + {len(quarantine)} cuarentena.",
    f"- Motivos de cuarentena: {quarantine['errores_bloqueantes'].value_counts().to_dict()}.", '',
    '## Archivos generados',
    f"- `{CONFIG['rutas']['notebook']}`",
    f"- `{CONFIG['rutas']['silver']}`",
    f"- `{CONFIG['rutas']['quarantine']}`",
    f"- `{CONFIG['rutas']['informe']}`", '',
    '## Reproducibilidad',
    f"- Fecha UTC: {EXECUTED_AT_UTC}.",
    f"- Python: {platform.python_version()}.",
    f"- pandas: {pd.__version__}.",
    '- Bronze JSON se lee sin modificar y se conserva el orden de los eventos.',
    '- Ejecutar las celdas en orden; los controles se realizan sobre los CSV persistidos.'
]
PATHS['informe'].write_text('\n'.join(report) + '\n', encoding='utf-8')
print('Informe generado:', PATHS['informe'])

Informe generado: C:\Users\remrodri\Github\practicasNotebookColab\proyecto-integradorV2\informes\bronze_silver\Informe_B2S_07_AndinaLog_Flota_Eventos.md
